In [ ]:
import anndata
import scvelo as scv
import pandas as pd
import os
import numpy as np
import numpy as np
import os
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import scanpy as sc
SEED = 2024
np.random.seed(SEED)
import unitvelo as utv
from itertools import chain

In [ ]:
files = ['1_celldancer', '2_scvelo_stochastic', '3_scvelo_dynamical','4_STT','5_unitvelo','6_DeepveloSA', '7_DeepveloGB','8_veloVI',"9_latentvelo","10_Dynamo_stochastic","11_Dynamo_deterministic","12_velocyto",'13_veloVAE','14_veloAE','15_biVI', '16_cell2fate', '17_DeepKINET','18_SDEvelo','20_TopicVelo']
#method_names = ['cellDancer', 'scvelo_stochastic', 'scvelo_dynamical', 'UniTvelo', 'velocity','DeepVelo_GB',"veloVI","LatentVelo","Dynamo_stochastic","Dynamo_deterministic","Velocyto"]
simuID_list = ['bifurcating_converging_seed1', 'bifurcating_converging_seed2', 'bifurcating_converging_seed3' ,'bifurcating_cycle_seed1', 'bifurcating_cycle_seed2', 'bifurcating_cycle_seed3' ,'bifurcating_loop_seed1', 'bifurcating_loop_seed2', 'bifurcating_loop_seed3' ,'bifurcating_seed1', 'bifurcating_seed2' ,'bifurcating_seed3' ,'binary_tree_seed1' ,'binary_tree_seed2', 'binary_tree_seed3', 'branching_seed1', 'branching_seed2', 'branching_seed3', 'consecutive_bifurcating_seed1', 'consecutive_bifurcating_seed2' ,'consecutive_bifurcating_seed3', 'converging_seed1', 'converging_seed2','converging_seed3', 'cycle_seed1', 'cycle_seed2', 'cycle_seed3' ,'cycle_simple_seed1' ,'cycle_simple_seed2', 'cycle_simple_seed3', 'disconnected_seed1' ,'disconnected_seed2', 'disconnected_seed3' ,'linear_seed1' ,'linear_seed2', 'linear_seed3', 'linear_simple_seed1' ,'linear_simple_seed2', 'linear_simple_seed3', 'trifurcating_seed1' ,'trifurcating_seed2' ,'trifurcating_seed3' ]


In [ ]:
cluster_edges = {"bifurcating_converging_seed1":[("sA_sB", "sB_sC"),("sA_sB", "sB_sD"),("sB_sC", "sC_sE"),("sB_sD", "sD_sE"),("sC_sE", "sE_sF"),("sD_sE", "sE_sF")],
                "bifurcating_converging_seed2":[("sA_sB", "sB_sC"),("sA_sB", "sB_sD"),("sB_sC", "sC_sE"),("sB_sD", "sD_sE"),("sC_sE", "sE_sF"),("sD_sE", "sE_sF")],
                "bifurcating_converging_seed3":[("sA_sB", "sB_sC"),("sA_sB", "sB_sD"),("sB_sC", "sC_sE"),("sB_sD", "sD_sE"),("sC_sE", "sE_sF"),("sD_sE", "sE_sF")],
                "bifurcating_cycle_seed1":[("sA_sB", "sB_sC"),("sA_sB", "sB_sD"),("sB_sC", "sC_sE"),("sB_sD", "sD_sE"),("sC_sE", "sE_sA"),("sD_sE", "sE_sA"),("sE_sA","sA_sB")],
                "bifurcating_cycle_seed2":[("sA_sB", "sB_sC"),("sA_sB", "sB_sD"),("sB_sC", "sC_sE"),("sB_sD", "sD_sE"),("sC_sE", "sE_sA"),("sD_sE", "sE_sA"),("sE_sA","sA_sB")],
                "bifurcating_cycle_seed3":[("sA_sB", "sB_sC"),("sA_sB", "sB_sD"),("sB_sC", "sC_sE"),("sB_sD", "sD_sE"),("sC_sE", "sE_sA"),("sD_sE", "sE_sA"),("sE_sA","sA_sB")],
                "bifurcating_loop_seed1":[("sA_sB", "sB_sC"),("sA_sB", "sB_sD"),("sB_sC", "sC_sA"),("sC_sA", "sA_sB")],
                "bifurcating_loop_seed2":[("sA_sB", "sB_sC"),("sA_sB", "sB_sD"),("sB_sC", "sC_sA"),("sC_sA", "sA_sB")],
                "bifurcating_loop_seed3":[("sA_sB", "sB_sC"),("sA_sB", "sB_sD"),("sB_sC", "sC_sA"),("sC_sA", "sA_sB")],
                "bifurcating_seed1":[("sA_sB", "sB_sBmid"),("sB_sBmid", "sBmid_sD"),("sB_sBmid", "sBmid_sC"),("sBmid_sC","sBmid_sD"),("sBmid_sC", "sC_sEndC"),("sBmid_sD", "sD_sEndD")],
                "bifurcating_seed2":[("sA_sB", "sB_sBmid"),("sB_sBmid", "sBmid_sD"),("sB_sBmid", "sBmid_sC"),("sBmid_sC","sBmid_sD"),("sBmid_sC", "sC_sEndC"),("sBmid_sD", "sD_sEndD")],
                "bifurcating_seed3":[("sA_sB", "sB_sBmid"),("sB_sBmid", "sBmid_sD"),("sB_sBmid", "sBmid_sC"),("sBmid_sC","sBmid_sD"),("sBmid_sC", "sC_sEndC"),("sBmid_sD", "sD_sEndD")],
                "binary_tree_seed1":[("sA_sB", "sB_sC"),("sB_sC","sC_sCmid"),("sC_sCmid", "sCmid_sD"),("sCmid_sD", "sD_sEndD"),("sC_sCmid", "sCmid_sE"),("sCmid_sE", "sE_sEmid"),("sE_sEmid","sEmid_sF"),("sEmid_sF","sF_sEndF"),("sE_sEmid","sEmid_sG"),("sEmid_sG","sG_sEndG")],
                "binary_tree_seed2":[("sA_sB", "sB_sC"),("sB_sC","sC_sCmid"),("sC_sCmid", "sCmid_sD"),("sCmid_sD", "sD_sEndD"),("sC_sCmid", "sCmid_sE"),("sCmid_sE", "sE_sEmid"),("sE_sEmid","sEmid_sF"),("sEmid_sF","sF_sEndF"),("sE_sEmid","sEmid_sG"),("sEmid_sG","sG_sEndG")],
                "binary_tree_seed3":[("sA_sB", "sB_sC"),("sB_sC","sC_sCmid"),("sC_sCmid", "sCmid_sD"),("sCmid_sD", "sD_sEndD"),("sC_sCmid", "sCmid_sE"),("sCmid_sE", "sE_sEndE"),("sCmid_sD","sCmid_sE")],
                "branching_seed1":[("sA_sB", "sB_sC"),("sB_sC","sC_sCmid"),("sC_sCmid","sCmid_sH"),("sCmid_sH","sH_sEndH"),("sC_sCmid","sCmid_sI"),("sCmid_sI","sI_sEndI"),("sC_sCmid","sCmid_sD"),("sCmid_sD","sD_sDmid"),("sD_sDmid","sDmid_sE"),("sDmid_sE","sE_sEndE"),("sD_sDmid","sDmid_sF"),("sDmid_sF","sF_sEndF"),("sD_sDmid","sDmid_sG"),("sDmid_sG","sG_sEndG")],
                "branching_seed2":[("sA_sB", "sB_sC"),("sB_sC","sC_sCmid"),("sC_sCmid","sCmid_sJ"),("sCmid_sJ","sJ_sEndJ"),("sC_sCmid","sCmid_sI"),("sCmid_sI","sI_sEndI"),("sC_sCmid","sCmid_sD"),("sCmid_sD","sD_sDmid"),("sD_sDmid","sDmid_sE"),("sDmid_sE","sE_sEndE"),("sD_sDmid","sDmid_sF"),("sDmid_sF","sF_sEndF"),("sD_sDmid","sDmid_sG"),("sDmid_sG","sG_sEndG"),("sD_sDmid","sDmid_sH"),("sDmid_sH","sH_sEndH")],
                "branching_seed3":[("sA_sB", "sB_sC"),("sB_sC","sC_sCmid"),("sC_sCmid","sCmid_sE"),("sCmid_sE","sE_sEndE"),("sC_sCmid","sCmid_sD"),("sCmid_sD","sD_sEndD")],
                "consecutive_bifurcating_seed1":[("sA_sB", "sB_sC"),("sB_sC","sC_sCmid"),("sC_sCmid", "sCmid_sD"),("sCmid_sD", "sD_sDmid"),("sC_sCmid", "sCmid_sE"),("sCmid_sE", "sE_sEndE"),("sD_sDmid","sDmid_sF"),("sDmid_sF","sF_sEndF"),("sD_sDmid","sDmid_sG"),("sDmid_sG","sG_sEndG")],
                "consecutive_bifurcating_seed2":[("sA_sB", "sB_sC"),("sB_sC","sC_sCmid"),("sC_sCmid", "sCmid_sD"),("sCmid_sD", "sD_sDmid"),("sC_sCmid", "sCmid_sE"),("sCmid_sE", "sE_sEndE"),("sD_sDmid","sDmid_sF"),("sDmid_sF","sF_sEndF"),("sD_sDmid","sDmid_sG"),("sDmid_sG","sG_sEndG")],
                "consecutive_bifurcating_seed3":[("sA_sB", "sB_sC"),("sB_sC","sC_sCmid"),("sC_sCmid", "sCmid_sD"),("sCmid_sD", "sD_sDmid"),("sC_sCmid", "sCmid_sE"),("sCmid_sE", "sE_sEndE"),("sD_sDmid","sDmid_sF"),("sDmid_sF","sF_sEndF"),("sD_sDmid","sDmid_sG"),("sDmid_sG","sG_sEndG")],
                "converging_seed1":[("preB_sB", "sB_sD"),("sB_sD","sD_sE"),("preC_sC", "sC_sD"),("sC_sD","sD_sE")],
                "converging_seed2":[("preB_sB", "sB_sD"),("sB_sD","sD_sE"),("preC_sC", "sC_sD"),("sC_sD","sD_sE")],
                "converging_seed3":[("preB_sB", "sB_sD"),("sB_sD","sD_sE"),("preC_sC", "sC_sD"),("sC_sD","sD_sE")],
                "cycle_seed1":[("s1_s2","s2_s3"),("s2_s3","s3_s1"),("s3_s1","s1_s2")],
                "cycle_seed2":[("s1_s2","s2_s3"),("s2_s3","s3_s1"),("s3_s1","s1_s2")],
                "cycle_seed3":[("s1_s2","s2_s3"),("s2_s3","s3_s1"),("s3_s1","s1_s2")],
                "cycle_simple_seed1":[("s1_s2","s2_s3"),("s2_s3","s3_s1"),("s3_s1","s1_s2")],
                "cycle_simple_seed2":[("s1_s2","s2_s3"),("s2_s3","s3_s1"),("s3_s1","s1_s2")],
                "cycle_simple_seed3":[("s1_s2","s2_s3"),("s2_s3","s3_s1"),("s3_s1","s1_s2")],
                "disconnected_seed1":[("right_s1_right_s2","right_s2_right_s3"),("right_s2_right_s3","right_s3_right_s1"),("right_s3_right_s1","right_s1_right_s2"),("left_sA_left_sB","left_sB_left_sC"),("left_sB_left_sC","left_sC_left_sEndC")],
                "disconnected_seed2":[("right_s1_right_s2","right_s2_right_s3"),("right_s2_right_s3","right_s3_right_s1"),("right_s3_right_s1","right_s1_right_s2"),("left_sA_left_sB","left_sB_left_sC"),("left_sB_left_sC","left_sC_left_sEndC")],
                "disconnected_seed3":[("right_s1_right_s2","right_s2_right_s3"),("right_s2_right_s3","right_s3_right_s1"),("right_s3_right_s1","right_s1_right_s2"),("left_sA_left_sB","left_sB_left_sC"),("left_sB_left_sC","left_sC_left_sEndC")],
                "linear_seed1":[("sA_sB", "sB_sC"),("sB_sC", "sC_sEndC")],
                "linear_seed2":[("sA_sB", "sB_sC"),("sB_sC", "sC_sEndC")],
                "linear_seed3":[("sA_sB", "sB_sC"),("sB_sC", "sC_sEndC")],
                "linear_simple_seed1":[("1","0")],
                "linear_simple_seed2":[("0","1"),("2","0")],
                "linear_simple_seed3":[("1","0"),("2","1")],
                "trifurcating_seed1":[("sA_sB", "sB_sC"),("sB_sC","sC_sCmid"),("sC_sCmid", "sCmid_sD"),("sCmid_sD", "sD_sEndD"),("sC_sCmid", "sCmid_sE"),("sCmid_sE", "sE_sEndE"),("sC_sCmid","sCmid_sF"),("sCmid_sF","sF_sEndF")],
                "trifurcating_seed2":[("sA_sB", "sB_sC"),("sB_sC","sC_sCmid"),("sC_sCmid", "sCmid_sD"),("sCmid_sD", "sD_sEndD"),("sC_sCmid", "sCmid_sE"),("sCmid_sE", "sE_sEndE"),("sC_sCmid","sCmid_sF"),("sCmid_sF","sF_sEndF")],
                "trifurcating_seed3":[("sA_sB", "sB_sC"),("sB_sC","sC_sCmid"),("sC_sCmid", "sCmid_sD"),("sCmid_sD", "sD_sEndD"),("sC_sCmid", "sCmid_sE"),("sCmid_sE", "sE_sEndE"),("sC_sCmid","sCmid_sF"),("sCmid_sF","sF_sEndF")],
                }

In [ ]:
def get_global_coherence(adata,label,basis_embedding,cluster_edges):
    ### read data. Note: the defalut velocity names in layers is 'velocity'.
    #adata = scv.read(adata_dir)
    ### calculate velocity
    #scv.pp.neighbors(adata, n_pcs=30, n_neighbors=30)
    #scv.pp.filter_and_normalize(adata, min_shared_counts=0, n_top_genes=2000)
    #scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
    #scv.tl.velocity(adata, mode='stochastic')
    ########## calculate embedding
    #scv.tl.velocity_graph(adata)
    #scv.tl.velocity_embedding(adata,basis = basis_embedding)
    #################### evaluate
    exp_metrics = utv.evaluate(adata, cluster_edges, label, 'velocity',x_emb = "X_" + basis_embedding)
    ###### global coherence
    in_cluster = exp_metrics["In-cluster Coherence"]
    cross_boundary = exp_metrics["Cross-Boundary Direction Correctness (A->B)"]
    in_res = np.median(list(chain.from_iterable(in_cluster.values())))
    cross_res =  np.median(list(chain.from_iterable(cross_boundary.values())))
    
    cell_number = adata.obs.shape[0]
    
    g1 = []
    for cluster0 in list(in_cluster.keys()):
        data_tmp1 = in_cluster[cluster0]
        p1 = len(data_tmp1)/cell_number
        h1 = np.mean(data_tmp1)
        
        d1 = 0
        count_num = 0
        for edge0 in list(cross_boundary.keys()):
            if cluster0 in edge0:
                ss = cross_boundary[edge0]
                count_num = count_num + 1
                d1 = d1 + np.sign(np.median(ss)) ########## sign
                #d1 <- d1 + np.median(ss) ########## median
        g1.append( p1 * h1 * d1 / count_num)
    return [in_res,cross_res,sum(g1)]

In [ ]:
############# correct 14_veloAE
for simuID in simuID_list:
    print(simuID+'\n')
    adata = scv.read("/media/kunyu/1-16T/cooperation/liyr/dyngen_new/anndata/"+simuID+"/anndata.h5ad")
    file2 = "/media/kunyu/1-16T/cooperation/liyr/dyngen_new/velocity/"+simuID+"/"+"14_veloAE"+"/"+"anndata.h5ad"
    if os.path.exists(file2):
        adata2 = scv.read(file2)
        #scv.pp.neighbors(adata)
        adata.layers['velocity'] = np.array(adata.layers['velocity'].todense())
        scv.tl.velocity_graph(adata)
        scv.tl.velocity_embedding(adata,basis = "pca")
        scv.pl.velocity_embedding_stream(adata,basis = "pca")
        adata.write_h5ad(file2)

In [ ]:
############# correct 9_latentvelo
for simuID in simuID_list:
    print(simuID+'\n')
    adata = scv.read("/media/kunyu/1-16T/cooperation/liyr/dyngen_new/anndata/"+simuID+"/anndata.h5ad")
    file2 = "/media/kunyu/1-16T/cooperation/liyr/dyngen_new/velocity/"+simuID+"/"+"9_latentvelo"+"/"+"anndata.h5ad"
    if os.path.exists(file2):
        adata2 = scv.read(file2)
        scv.pp.neighbors(adata)
        adata.layers['velocity'] = np.array(adata.layers['velocity'].todense())
        scv.tl.velocity_graph(adata)
        scv.tl.velocity_embedding(adata,basis = "pca")
        scv.pl.velocity_embedding_stream(adata,basis = "pca")
        adata.write_h5ad(file2)

In [ ]:
############# correct 13_veloVAE
for simuID in simuID_list:
    print(simuID+'\n')
    adata = scv.read("/media/kunyu/1-16T/cooperation/liyr/dyngen_new/anndata/"+simuID+"/anndata.h5ad")
    file2 = "/media/kunyu/1-16T/cooperation/liyr/dyngen_new/velocity/"+simuID+"/"+"13_veloVAE"+"/"+"anndata.h5ad"
    if os.path.exists(file2):
        adata2 = scv.read(file2)
        scv.pp.neighbors(adata)
        adata.layers['velocity'] = np.array(adata.layers['velocity'].todense())
        scv.tl.velocity_graph(adata)
        scv.tl.velocity_embedding(adata,basis = "pca")
        scv.pl.velocity_embedding_stream(adata,basis = "pca")
        adata.write_h5ad(file2)

In [ ]:
############################## in, cross, global
in_list = {}
cross_list = {}
global_list = {}
label  = "clusters"
basis_embedding = "pca"

for simuID in simuID_list:
    in_list_tmp = {}
    cross_list_tmp = {}
    global_list_tmp = {}
    
    for i in range(len(files)):
    #for i in [4]:
        print("\n")
        print(simuID +"_"+ files[i])
        print("\n")
        file2 = "/media/kunyu/1-16T/cooperation/liyr/dyngen_new/velocity/"+simuID+"/"+files[i]+"/"+"anndata.h5ad"
        if os.path.exists(file2):        
            adata = scv.read(file2)
            
            res  = get_global_coherence(adata,label,basis_embedding,cluster_edges[simuID])
            in_list_tmp[files[i]] = res[0]
            cross_list_tmp[files[i]] = res[1]
            global_list_tmp[files[i]] = res[2]
        else:
            in_list_tmp[files[i]] = np.nan
            cross_list_tmp[files[i]] = np.nan
            global_list_tmp[files[i]] = np.nan
    in_list[simuID] = in_list_tmp
    cross_list[simuID] = cross_list_tmp
    global_list[simuID] = global_list_tmp

In [ ]:
pd.DataFrame(in_list).to_csv("eval/eval_incluster.csv")
pd.DataFrame(cross_list).to_csv("eval/eval_direction.csv")
pd.DataFrame(global_list).to_csv("eval/eval_global.csv")